# Anggota 3 (ML Engineer) — Retrain dengan Data Realistis

**Tujuan:** Melatih ulang Custom CNN menggunakan dataset sintetis diperkaya augmentasi realistis (zoom-out, miring, cahaya beragam, bayangan, blur) agar model akurat di kondisi kamera nyata.

**Format input wajib:**
- 64x64 piksel, grayscale
- Titik **GELAP** di latar **TERANG** (bukan biner putih-di-hitam)
- 1 sel per gambar, ada margin

In [ ]:
# Mount Google Drive dan Ekstrak Dataset
from google.colab import drive
import zipfile, os

print('Mounting Google Drive...')
drive.mount('/content/drive')

if not os.path.exists('/content/dataset_synthetic'):
    print('Mengekstrak dataset_synthetic.zip dari Google Drive...')
    with zipfile.ZipFile('/content/drive/MyDrive/dataset_synthetic.zip', 'r') as z:
        z.extractall('/content/')
    print('Dataset selesai diekstrak.')

print('Setup Colab selesai!')

In [ ]:
!pip install kagglehub --quiet

import os, shutil, random, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {tf.config.list_physical_devices("GPU")}')
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## Load Dataset

In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/content/dataset_synthetic')
print(f'Dataset: {DATASET_ROOT}')
for item in sorted(DATASET_ROOT.iterdir()):
    count = len(list(item.glob('*.*')))
    print(f'  {item.name.upper()}: {count} file')

## Konfigurasi Hyperparameter

In [ ]:
IMG_SIZE    = 64
BATCH_SIZE  = 32
NUM_CLASSES = 26
EPOCHS      = 100
LR          = 5e-4

all_paths, all_labels = [], []
subdirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])

for cls_dir in subdirs:
    char = cls_dir.name
    imgs = list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpg'))
    for img in imgs:
        all_paths.append(str(img))
        all_labels.append(char)

unique_classes = sorted(set(all_labels))
class_to_idx  = {cls: i for i, cls in enumerate(unique_classes)}
idx_to_class  = {i: cls for cls, i in class_to_idx.items()}
int_labels    = [class_to_idx[l] for l in all_labels]

print(f'Total gambar    : {len(all_paths)}')
print(f'Total kelas     : {len(unique_classes)}')

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, int_labels, test_size=0.30, random_state=SEED, stratify=int_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels
)
print(f'   Train : {len(train_paths)} | Val : {len(val_paths)} | Test : {len(test_paths)}')

## tf.data Pipeline + Augmentasi REALISTIS (Revisi Anggota 3)

Augmentasi yang ditambahkan/diperbesar sesuai daftar revisi:
- **Zoom-out 50%** (titik kecil dari jauh) — dari sebelumnya hanya 10%
- **Kemiringan 25°** (tangan gemetar) — dari sebelumnya hanya 11°
- **Translasi 15%** (huruf di tepi frame) — dari sebelumnya hanya 5%
- **Brightness acak 40%** (cahaya redup/silau/bayangan jari)
- **Gaussian Noise** (simulasi sensor kamera HP mid-range)

**Aturan wajib format input model:**
Grayscale, titik GELAP di latar TERANG. BUKAN gambar biner threshold.

In [ ]:
def load_and_preprocess(img_path, label):
    """
    Load gambar -> grayscale -> resize -> normalisasi ke [0,1].
    FORMAT WAJIB: titik GELAP di latar TERANG, grayscale, 64x64.
    Model TIDAK menerima gambar biner (output Adaptive Threshold).
    """
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='nearest')
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.grayscale_to_rgb(img)
    return img, label


# AUGMENTASI REALISTIS YANG DIPERLUAS
# Dibandingkan training sebelumnya, parameter berikut DIPERBESAR agar model
# kebal terhadap variasi kondisi kamera nyata.
augmentation = keras.Sequential([
    # 1. Kemiringan: +-25 derajat (dari sebelumnya hanya +-11 derajat)
    #    Simulasi: HP miring / tangan tidak stabil saat memegang
    layers.RandomRotation(0.07),

    # 2. Zoom: dari 50% zoom-out hingga 20% zoom-in (dari sebelumnya hanya +-10%)
    #    Simulasi: jarak HP ke kertas bervariasi (sangat dekat s/d agak jauh)
    layers.RandomZoom((-0.5, 0.2)),

    # 3. Translasi: +-15% (dari sebelumnya hanya +-5%)
    #    Simulasi: huruf tidak selalu tepat di tengah frame, bisa ke tepi
    layers.RandomTranslation(0.15, 0.15),

    # 4. Kontras acak: lebih lebar (dari sebelumnya 0.2)
    #    Simulasi: kondisi cahaya redup / terlalu terang / bayangan jari
    layers.RandomContrast(0.5, value_range=(0.0, 1.0)),

    # 5. Brightness acak
    #    Simulasi: lampu berbeda-beda, ada bayangan, ada silau dari jendela
    layers.RandomBrightness(0.4, value_range=(0.0, 1.0)),

    # 6. Gaussian Noise (simulasi sensor kamera HP mid-range)
    #    Noise kecil membantu model tidak terlalu bergantung pada tekstur halus
    layers.Lambda(lambda x: x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.05)),
    layers.Lambda(lambda x: tf.clip_by_value(x, 0.0, 1.0)),

], name='augmentation_realistis')

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
            .shuffle(len(train_paths), seed=SEED)
            .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
            .map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds   = (tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
            .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

test_ds  = (tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
            .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

print('tf.data pipeline realistis siap!')
print(f'  Train batches : {len(train_ds)}')
print(f'  Val   batches : {len(val_ds)}')
print(f'  Test  batches : {len(test_ds)}')

## Visualisasi Augmentasi Realistis

Periksa hasil augmentasi sebelum training. Pastikan titik masih terlihat GELAP di latar TERANG meski sudah dirotasi/zoom/noise.

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle('Contoh Augmentasi Realistis — 1 Huruf Diolah Jadi 6 Variasi', fontsize=13, fontweight='bold')

for col, (img_path, lbl) in enumerate(zip(train_paths[:6], train_labels[:6])):
    orig = tf.image.decode_image(tf.io.read_file(img_path), channels=1)
    orig = tf.image.resize(orig, [IMG_SIZE, IMG_SIZE], method='nearest')
    orig = tf.cast(orig, tf.float32) / 255.0
    orig_rgb = tf.image.grayscale_to_rgb(orig)

    axes[0, col].imshow(tf.squeeze(orig).numpy(), cmap='gray')
    char_name = idx_to_class[lbl].upper()
    axes[0, col].set_title(f'Asli: {char_name}', fontsize=9)
    axes[0, col].axis('off')

    aug1 = augmentation(orig_rgb, training=True)
    aug2 = augmentation(orig_rgb, training=True)

    axes[1, col].imshow(tf.reduce_mean(aug1, axis=-1).numpy(), cmap='gray')
    axes[1, col].set_title('Variasi #1', fontsize=9)
    axes[1, col].axis('off')

    axes[2, col].imshow(tf.reduce_mean(aug2, axis=-1).numpy(), cmap='gray')
    axes[2, col].set_title('Variasi #2', fontsize=9)
    axes[2, col].axis('off')

plt.tight_layout()
plt.show()
print('Pastikan titik Braille masih terlihat (meski blur/miring/kecil/noise)!')

## Arsitektur Custom CNN

In [ ]:
def build_braille_cnn(num_classes=26, img_size=64):
    inputs = keras.Input(shape=(img_size, img_size, 3), name='input')

    x = layers.Conv2D(32, 3, padding='same', use_bias=False, name='conv1a')(inputs)
    x = layers.BatchNormalization(name='bn1a')(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False, name='conv1b')(x)
    x = layers.BatchNormalization(name='bn1b')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D(2, name='pool1')(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Conv2D(64, 3, padding='same', use_bias=False, name='conv2a')(x)
    x = layers.BatchNormalization(name='bn2a')(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, padding='same', use_bias=False, name='conv2b')(x)
    x = layers.BatchNormalization(name='bn2b')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D(2, name='pool2')(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(128, 3, padding='same', use_bias=False, name='conv3a')(x)
    x = layers.BatchNormalization(name='bn3a')(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False, name='conv3b')(x)
    x = layers.BatchNormalization(name='bn3b')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D(2, name='pool3')(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(256, 3, padding='same', use_bias=False, name='conv4a')(x)
    x = layers.BatchNormalization(name='bn4a')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool2D(2, name='pool4')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(num_classes, activation='softmax', name='output')(x)

    return keras.Model(inputs, x, name='BrailleCNN_Realistic')

model = build_braille_cnn(NUM_CLASSES, IMG_SIZE)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_acc')]
)
model.summary()

---
## Training Model — Retrain Realistis

In [ ]:
os.makedirs('/content/drive/MyDrive/BrailleVision_Output_Realistic/checkpoints', exist_ok=True)
checkpoint_path = '/content/drive/MyDrive/BrailleVision_Output_Realistic/checkpoints/braille_cnn_realistic_best.keras'

callbacks = [
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-7,
        verbose=1
    )
]

print('Memulai Retrain Custom CNN — Augmentasi Realistis')
print(f'   Input  : {IMG_SIZE}x{IMG_SIZE}px grayscale titik-gelap-di-terang -> 3ch')
print(f'   LR     : {LR}  (lebih kecil dari training awal 1e-3)')
print(f'   Epochs : hingga {EPOCHS} (EarlyStopping aktif)')
print('   Augment: Zoom +-50%, Rotasi +-25deg, Brightness +-40%, Gaussian Noise')
print('=' * 60)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1
)

best_val = max(history.history['val_accuracy'])
print(f'\nTraining selesai! Best Val Accuracy: {best_val*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History — BrailleVision Custom CNN (Realistis)', fontsize=13, fontweight='bold')

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].axhline(0.85, color='green', linestyle='--', label='Target 85%')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history.history['top3_acc'], label='Train Top-3')
axes[2].plot(history.history['val_top3_acc'], label='Val Top-3')
axes[2].set_title('Top-3 Accuracy'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/BrailleVision_Output_Realistic/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## Evaluasi Model — Test Set

In [ ]:
print('Evaluasi pada Test Set...')
test_loss, test_acc, test_top3 = model.evaluate(test_ds, verbose=0)

print(f'  Test Loss      : {test_loss:.4f}')
print(f'  Test Accuracy  : {test_acc*100:.2f}%')
print(f'  Test Top-3 Acc : {test_top3*100:.2f}%')

if test_acc >= 0.85:
    print('Target akurasi >=85% TERCAPAI!')
else:
    diff = 0.85 - test_acc
    print(f'Akurasi {test_acc*100:.2f}% — kurang {diff*100:.2f}% dari target.')
    print('Pertimbangkan: tambah data kamera nyata atau kurangi Dropout.')

y_true, y_pred_probs = [], []
for imgs, labels in test_ds:
    preds = model(imgs, training=False)
    y_true.extend(labels.numpy())
    y_pred_probs.extend(preds.numpy())

y_pred = np.argmax(y_pred_probs, axis=1)
class_name_labels = [idx_to_class[i].upper() for i in range(NUM_CLASSES)]

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_name_labels))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_name_labels, yticklabels=class_name_labels,
            linewidths=0.5, ax=ax)
ax.set_title(f'Confusion Matrix (Acc: {test_acc*100:.2f}%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/BrailleVision_Output_Realistic/confusion_matrix_realistic.png', dpi=150, bbox_inches='tight')
plt.show()

## Simpan Model dan Konversi ke TFLite

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/BrailleVision_Output_Realistic'
os.makedirs(SAVE_DIR, exist_ok=True)

model_keras_path = f'{SAVE_DIR}/braille_vision_model_realistic.keras'
model.save(model_keras_path)
size_mb = os.path.getsize(model_keras_path) / (1024**2)
print(f'Model Keras tersimpan: {size_mb:.2f} MB')

label_map_out = {str(i): cls.upper() for i, cls in idx_to_class.items()}
with open(f'{SAVE_DIR}/label_map_realistic.json', 'w') as f:
    json.dump(label_map_out, f, indent=2)
print('Label map tersimpan.')

print('Konversi ke TFLite Float16...')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_f16 = converter.convert()
tflite_f16_path = f'{SAVE_DIR}/braille_vision_f16_realistic.tflite'
with open(tflite_f16_path, 'wb') as f:
    f.write(tflite_f16)
size_f16 = os.path.getsize(tflite_f16_path) / (1024**2)
print(f'TFLite Float16: {size_f16:.2f} MB')

print('Konversi ke TFLite Int8...')
def representative_dataset():
    for imgs, _ in train_ds.take(50):
        for img in imgs:
            yield [tf.expand_dims(img, 0)]

converter2 = tf.lite.TFLiteConverter.from_keras_model(model)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]
converter2.representative_dataset = representative_dataset
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter2.inference_input_type  = tf.int8
converter2.inference_output_type = tf.int8
tflite_int8 = converter2.convert()
tflite_int8_path = f'{SAVE_DIR}/braille_vision_int8_realistic.tflite'
with open(tflite_int8_path, 'wb') as f:
    f.write(tflite_int8)
size_int8 = os.path.getsize(tflite_int8_path) / (1024**2)
print(f'TFLite Int8: {size_int8:.2f} MB')

print('Ringkasan Ukuran:')
print(f'  Keras     : {size_mb:.2f} MB')
print(f'  Float16   : {size_f16:.2f} MB')
print(f'  Int8      : {size_int8:.2f} MB')

## Validasi TFLite dan Download

In [ ]:
print('Memvalidasi akurasi TFLite Float16 pada test set...')

interpreter = tf.lite.Interpreter(model_path=tflite_f16_path)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

correct, total = 0, 0
for imgs, labels in test_ds:
    for img, label in zip(imgs, labels):
        inp = tf.expand_dims(img, 0)
        interpreter.set_tensor(input_details[0]['index'], inp)
        interpreter.invoke()
        pred = np.argmax(interpreter.get_tensor(output_details[0]['index']))
        if pred == label.numpy():
            correct += 1
        total += 1

tflite_acc = correct / total
print(f'  TFLite Float16 Accuracy : {tflite_acc*100:.2f}%')
print(f'  Keras model   Accuracy  : {test_acc*100:.2f}%')
print(f'  Delta                   : {abs(test_acc - tflite_acc)*100:.2f}%')

if tflite_acc >= 0.85:
    print('Target >=85% TERCAPAI pada TFLite! Siap dikirim ke Anggota 5.')
else:
    print(f'Akurasi TFLite {tflite_acc*100:.2f}% — belum 85%. Pertimbangkan tambah data kamera nyata.')

from google.colab import files
print('Mengunduh file model ke komputer Anda...')
files.download(tflite_f16_path)
files.download(tflite_int8_path)
files.download(f'{SAVE_DIR}/label_map_realistic.json')
print('Selesai! File siap diserahkan ke Anggota 5 (Integration Engineer).')